# 154 — Costo, latencia, caching y capacidad

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


## 📖 Materia resumida

La economía de la inferencia se gobierna con cuatro instrumentos:

- **Costo por token**: `costo = t_in·p_in + t_out·p_out`, con la salida más
  cara que la entrada. Es lineal: el prompt de sistema se paga en *cada*
  petición, y `max_tokens` acota el peor caso.
- **Latencia por percentiles**: TTFT (procesar el prompt) + tiempo por token
  de salida. Se reporta p50/p95/p99, nunca promedio: la distribución tiene
  cola pesada y, con varias llamadas por sesión, casi todo usuario toca la
  cola (10 llamadas → P(al menos una > p95) = 1 − 0.95¹⁰ ≈ 40 %).
- **Caching**: exacta (hash del prompt), *prompt caching* del prefijo (KV
  cache, precio reducido) y semántica (por similitud — con riesgo de falsos
  aciertos, que son errores de calidad silenciosos).
  `costo_efectivo ≈ (1−h)·costo + h·costo_lookup`.
- **Capacidad**: ley de Little `L = λ·W`; utilización `ρ = λ·S/c` lejos de 1
  (la espera en cola crece de forma no lineal al saturar); dimensionar el pico
  con margen y definir degradación explícita antes del rechazo.


## 🧮 Ejemplo de referencia

200 000 peticiones/mes; 1 800 tokens de entrada (1 200 de prefijo estable),
350 de salida; p_in = 3, p_out = 15, prefijo cacheado = 0.3 USD/MTok, tasa de
acierto 0.9.

```text
Sin caché:   0.0054 + 0.00525            = 0.01065 USD/petición → 2 130 USD/mes
Con caché:   0.0018 + 0.000324 + 0.00036 = entrada 0.002484
             0.002484 + 0.00525          = 0.007734 USD/petición → 1 547 USD/mes  (−27 %)
```

Capacidad: λ = 5 req/s, S = 2 s, 4 concurrentes por réplica → L = 10;
3 réplicas darían ρ ≈ 0.83 (demasiado justo); con margen ρ ≤ 0.6 se
dimensionan 5 réplicas, y la decisión se documenta con sus supuestos.


In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


In [ ]:
result = run_lab("observability", seed=154)
show(result)


## Reflexión

1. Tu servicio reporta latencia promedio de 900 ms y los usuarios se quejan de
   lentitud: ¿qué percentiles pedirías y por qué el promedio puede ocultar el
   problema si cada sesión hace varias llamadas?
2. La caché semántica subió su tasa de aciertos del 20 % al 45 % tras bajar el
   umbral de similitud: ¿qué métrica adicional necesitas antes de celebrar el
   ahorro y cómo la medirías?
3. Si duplicas las réplicas para bajar la espera en cola, ¿qué parte del costo
   por petición NO cambia y qué palanca tendrías que tocar para reducirla?
